## Advanced_CI_w4
### CH6

### 6.3 회귀분석으로 CATE 구하기

In [1]:
import pandas as pd
import numpy as np

data = pd.read_csv("daily_restaurant_sales.csv")
data.head()

,rest_id,day,month,weekday,weekend,is_holiday,is_dec,is_nov,competitors_price,discounts,sales
0,0,2016-01-01,1,4,False,True,False,False,2.88,0,79.0
1,0,2016-01-02,1,5,True,False,False,False,2.64,0,57.0
2,0,2016-01-03,1,6,True,False,False,False,2.08,5,294.0
3,0,2016-01-04,1,0,False,False,False,False,3.37,15,676.5
4,0,2016-01-05,1,1,False,False,False,False,3.79,0,66.0


In [2]:
import statsmodels.formula.api as smf

X = ["C(month)", "C(weekday)", "is_holiday", "competitors_price"]
regr_cate = smf.ols(f"sales ~ discounts*({'+'.join(X)})", data = data).fit()

In [3]:
ols_cate_pred = (
    regr_cate.predict(data.assign(discounts=data["discounts"]+1)) - regr_cate.predict(data)
)

### 6.4 CATE 예측 평가하기

In [4]:
train = data.query("day<'2018-01-01'")
test = data.query("day>='2018-01-01'")

In [13]:
X = ["C(month)", "C(weekday)", "is_holiday", "competitors_price"]
regr_model = smf.ols(f"sales ~ discounts * ({'+'.join(X)})", data = train).fit()

cate_pred =(
    regr_model.predict(test.assign(discounts=test["discounts"]+1)) - regr_model.predict(test)
)

In [14]:
from sklearn.ensemble import GradientBoostingRegressor

X = ["month", "weekday", "is_holiday", "competitors_price", "discounts"]
y = "sales"

np.random.seed(1)
ml_model = GradientBoostingRegressor(n_estimators = 50).fit(train[X], train[y])
ml_pred = ml_model.predict(test[X])

In [15]:
np.random.seed(123)

test_pred = test.assign(
    ml_pred = ml_pred,
    cate_pred = cate_pred,
    rand_m_pred = np.random.uniform(-1, 1, len(test)),
)

### 6.5 분위수 효과 곡선

In [16]:
from toolz import curry

@curry
def effect(data, y, t):
    return (np.sum((data[t] - data[t].mean())*data[y]) / np.sum((data[t] - data[t].mean())**2))

In [17]:
float(effect(test, "sales", "discounts"))

32.16196368039615

In [18]:
def effect_by_quntile(df, pred, y, t, q=10):
    groups = np.round(pd.IntervalIndex(pd.qcut(df[pred], q=q)).mid, 2) # 중복값 제거
    return (df
            .assign(**{f"{pred}_quantile": groups})
            .groupby(f"{pred}_quantile")
            .apply(effect(y=y, t=t)))
effect_by_quntile(test_pred, "cate_pred", y="sales", t="discounts")

C:\Users\User\AppData\Local\Temp\ipykernel_35760\2765057384.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(effect(y=y, t=t)))


cate_pred_quantile
17.50    20.494153
23.93    24.782101
26.85    27.494156
28.95    28.833993
30.81    29.604257
32.68    32.216500
34.65    35.889459
36.75    36.846889
39.40    39.125449
47.36    44.272549
dtype: float64

In [ ]:
# 누적 표본에 따른 효과

def cumulative_effect_curve(dataset, prediction, y, t, ascending=False, steps=100):
    size = len(dataset)
    ordered_df = dataset.sort_values(prediction, ascending=ascending).reset_index(drop=True)
    steps = np.linspace(size/steps, size, steps).round(0)
    return np.array([effect(ordered_df.query(f"index<={row}"), y=y, t=t) for row in steps])
cumulative_effect_curve(test_pred, "cate_pred", y="sales", t="discounts")

array([49.65116279, 49.37712454, 46.20360341, 45.9770684 , 45.41694622,
       45.23107566, 44.91075165, 44.68841405, 44.56309576, 44.27254859,
       43.95937553, 43.66601716, 43.30710192, 42.71556602, 42.57717619,
       42.14190207, 41.9120469 , 32.15945589, 41.59626585, 41.38073714,
       41.11778111, 41.04864418, 40.95933655, 40.6949357 , 40.5585599 ,
       40.32182723, 40.19431251, 40.0242349 , 39.92014581, 39.58537488,
       39.44592589, 39.53382731, 39.41985355, 39.29287716, 39.27763551,
       39.14056914, 39.04402217, 38.98421346, 38.9022713 , 38.71601845,
       38.72423427, 38.53676256, 38.56709164, 38.36661748, 38.27685106,
       38.06110588, 37.99479087, 37.89750017, 37.72699021, 37.55580363,
       37.35825912, 37.28006651, 37.21139791, 37.06210857, 37.00753334,
       36.87259686, 36.74299254, 36.52014459, 36.4873419 , 36.38463167,
       36.21632077, 36.10908232, 35.90865968, 35.73915746, 35.56123101,
       35.42248759, 35.37393053, 35.25994908, 35.24501879, 35.14

In [26]:
# 누적 이득 곡선
def cumulative_gain_curve(df, prediction, y, t, ascending=False, normalize=False, steps=100):
    effect_fn = effect(y=y, t=t)
    normalizer = effect_fn(df) if normalize else 0
    size = len(df)
    ordered_df = df.sort_values(prediction, ascending=ascending).reset_index(drop=True)
    steps = np.linspace(size/steps, size, steps).round(0)
    effects = [(effect_fn(ordered_df.query(f"index<={row}"))-normalizer)*(row/size) for row in steps]
    return np.array([0]+effects)
cumulative_gain_curve(test_pred, "cate_pred", y="sales", t="discounts")

array([ 0.        ,  0.50387597,  0.982917  ,  1.38863289,  1.83046877,
        2.2690746 ,  2.71880783,  3.13779256,  3.57577084,  4.01798404,
        4.42379877,  4.83861979,  5.23242282,  5.62890903,  5.98551452,
        6.38159081,  6.74402024,  7.13257317,  7.5088165 ,  7.9068624 ,
        8.26968674,  8.63441305,  9.03646968,  9.41649072,  9.76869065,
       10.13172457, 10.48178647, 10.85654343, 11.20116176, 11.57715392,
       11.88179285, 12.22484979, 12.65329367, 13.00147394, 13.35835129,
       13.75177168, 14.08571614, 14.44720258, 14.8206963 , 15.16915263,
       15.48942971, 15.87058782, 16.18483861, 16.5889676 , 16.87711862,
       17.226077  , 17.51523959, 17.85547549, 18.1943502 , 18.48062947,
       18.77790182, 19.05825319, 19.38214231, 19.7240743 , 20.00659491,
       20.35269886, 20.65268403, 20.93862963, 21.18225404, 21.53351325,
       21.82793867, 22.09450014, 22.38142964, 22.62161465, 22.87752468,
       23.11063608, 23.3799479 , 23.70688475, 23.97456335, 24.32

### 6.8 목표 변환

In [28]:
X = ["C(month)", "C(weekday)", "is_holiday", "competitors_price"]

y_res = smf.ols(f"sales ~ {'+'.join(X)}", data = test).fit().resid
t_res = smf.ols(f"discounts ~ {'+'.join(X)}", data = test).fit().resid

tau_hat = y_res / t_res

In [29]:
from sklearn.metrics import mean_squared_error

for m in ["rand_m_pred", "ml_pred", "cate_pred"]:
     wmse = mean_squared_error(tau_hat, test_pred[m], sample_weight=t_res**2)
     print(f"MSE for {m}:", wmse)

MSE for rand_m_pred: 1115.803515760459
MSE for ml_pred: 576256.7425385388
MSE for cate_pred: 42.90447405550283
